# 12 因果推論 — 參考解答

松柏護理之家退伍軍人症群聚事件因果推論練習的完整解答。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import statsmodels.formula.api as smf

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["died"] = (df["outcome"] == "dead").astype(int)

## 題目 1：水療暴露的歸因風險

In [ ]:
# 水療暴露
hydro_exp = df[df["hydrotherapy_use"] == 1]
hydro_unexp = df[df["hydrotherapy_use"] == 0]

risk_hydro_exp = hydro_exp["infected"].mean()
risk_hydro_unexp = hydro_unexp["infected"].mean()
risk_total = df["infected"].mean()

AR_hydro = risk_hydro_exp - risk_hydro_unexp
PAR_hydro = risk_total - risk_hydro_unexp
PAR_pct_hydro = PAR_hydro / risk_total * 100

print("=== 水療暴露 ===")
print(f"使用者侵襲率：{risk_hydro_exp:.1%}")
print(f"非使用者侵襲率：{risk_hydro_unexp:.1%}")
print(f"AR = {AR_hydro:.3f}")
print(f"PAR% = {PAR_pct_hydro:.1f}%")

# 淋浴暴露（比較用）
shower_exp = df[df["shower_use"] == 1]
shower_unexp = df[df["shower_use"] == 0]
risk_sh_exp = shower_exp["infected"].mean()
risk_sh_unexp = shower_unexp["infected"].mean()
AR_shower = risk_sh_exp - risk_sh_unexp
PAR_shower = risk_total - risk_sh_unexp
PAR_pct_shower = PAR_shower / risk_total * 100

print(f"\n=== 淋浴暴露（比較）===")
print(f"AR = {AR_shower:.3f}")
print(f"PAR% = {PAR_pct_shower:.1f}%")

print(f"\n=== 比較 ===")
if abs(AR_shower) > abs(AR_hydro):
    print("\u2192 淋浴暴露的 AR 較大，對感染的貢獻更高")
else:
    print("\u2192 水療暴露的 AR 較大")

print("\n\u2192 AR 代表『如果因果關係成立，消除暴露可減少的風險量』")
print("\u2192 前提：(1) 因果關係成立 (2) 無干擾因子 (3) 暴露是可消除的")

## 題目 2：改變 DiD 介入日期

In [ ]:
cases = df[df["infected"] == 1].copy()
all_dates = pd.date_range("2026-01-12", "2026-01-28", freq="D")

# 介入組 / 對照組
treated_mask = (cases["floor"].isin([2, 3])) & (cases["wing"] == "B")
treated_daily = cases[treated_mask].groupby("symptom_onset_date").size().reindex(all_dates, fill_value=0)
control_daily = cases[~treated_mask].groupby("symptom_onset_date").size().reindex(all_dates, fill_value=0)

# 比較不同介入日
for cutoff in ["2026-01-22", "2026-01-25"]:
    panel = pd.DataFrame({
        "date": list(all_dates) * 2,
        "treated": [1] * len(all_dates) + [0] * len(all_dates),
        "daily_cases": list(treated_daily.values) + list(control_daily.values),
    })
    panel["post"] = (panel["date"] >= cutoff).astype(int)

    model = smf.ols("daily_cases ~ treated + post + treated:post", data=panel).fit()
    coef = model.params["treated:post"]
    pval = model.pvalues["treated:post"]

    print(f"介入日 = {cutoff}: treated:post = {coef:.3f}, p = {pval:.4f}")

print("\n\u2192 改變介入日期會影響 DiD 結果")
print("\u2192 原因：介入前後的觀察天數不同，前後病例數分布也不同")
print("\u2192 選擇介入日必須基於實際事件（真的消毒了），不能隨意挑選")
print("\u2192 如果亂挑介入日做出顯著結果 = p-hacking")

## 題目 3（挑戰題）：碰撞因子偏誤的實證

In [ ]:
from epi_learning import risk_ratio

# 全體 RR
ct_all = pd.crosstab(df["shower_use"], df["infected"])
rr_all = risk_ratio(ct_all.iloc[1, 1], ct_all.iloc[1].sum(), ct_all.iloc[0, 1], ct_all.iloc[0].sum())
print(f"=== 全體 RR (shower \u2192 infected) ===")
print(f"RR = {rr_all:.3f}")

# 只取住院者
hosp = df[df["hospitalized"] == 1].copy()
print(f"\n住院者：{len(hosp)} 人")
print(f"住院者中 shower_use 分布：{hosp['shower_use'].value_counts().to_dict()}")
print(f"住院者中 infected 分布：{hosp['infected'].value_counts().to_dict()}")

# 住院者全部都是感染者嗎？
if hosp["infected"].nunique() == 1:
    print("\n\u2192 住院者全部都是感染者（infected=1），無法計算 RR")
    print("\u2192 這正是碰撞因子偏誤的極端情況！")
    print("\u2192 因為只有感染且嚴重的人才會住院")
    print("\u2192 在住院者中，shower_use 和 infected 的關係被扭曲")
else:
    ct_hosp = pd.crosstab(hosp["shower_use"], hosp["infected"])
    rr_hosp = risk_ratio(ct_hosp.iloc[1, 1], ct_hosp.iloc[1].sum(), ct_hosp.iloc[0, 1], ct_hosp.iloc[0].sum())
    print(f"住院者 RR = {rr_hosp:.3f}")
    print(f"全體 RR = {rr_all:.3f}")
    print(f"\n\u2192 限定住院者後 RR 改變了！")
    print("\u2192 這就是碰撞因子偏誤（collider bias）")

print("\n=== 碰撞因子偏誤解釋 ===")
print("hospitalized \u2190 severity \u2190 infection")
print("hospitalized \u2190 infection")
print("\u2192 hospitalized 是碰撞因子，受 severity 和 infection 共同影響")
print("\u2192 條件化碰撞因子（只看住院者）= 打開一條假性路徑")
print("\u2192 結果：在住院者中，shower_use 和 infection 的關係被扭曲")

### 解讀

- **AR/PAR**：水療 vs 淋浴的歸因風險不同，反映不同暴露途徑的貢獻。淋浴是產生退伍軍人菌氣溶膠的主要途徑
- **DiD 介入日**：結果對介入日期敏感。正確的做法是用實際介入日期，不是事後挑選最顯著的
- **碰撞因子**：只分析住院者 = 對碰撞因子做條件化，會產生 selection bias。這是觀察性研究中常見的陷阱
- **因果推論的限制**：在觀察性資料中，我們永遠無法完全確定因果關係。DAG 和統計方法只能幫我們辨識和減少偏誤，但不能消除所有未觀測到的干擾因子

## 題目 4 解答

In [ ]:
# 疫苗接種率政策 DiD：部分行政區推行加強接種活動（treated），比較政策前後發生率
rng = np.random.default_rng(1204)
_rows = []
TRUE_EFFECT = -8.0   # 政策真正讓發生率下降 8/10萬
for dz in range(200):
    treated = 1 if dz < 100 else 0
    base = rng.normal(45, 6)
    for post in (0, 1):
        inc = base - 3 * post + TRUE_EFFECT * (treated * post) + rng.normal(0, 4)
        _rows.append({"district": dz, "treated": treated, "post": post, "incidence": inc})
vax = pd.DataFrame(_rows)
print(f"DiD 資料：{vax['district'].nunique()} 區 × 2 期，真值效果 = {TRUE_EFFECT}/10萬")

m = vax.groupby(["treated", "post"])["incidence"].mean().unstack()
did_manual = (m.loc[1, 1] - m.loc[1, 0]) - (m.loc[0, 1] - m.loc[0, 0])
fit = smf.ols("incidence ~ treated * post", vax).fit()
print(m.round(2))
print(f"\n手算 DiD = {did_manual:.2f}")
print(f"迴歸交互項 treated:post = {fit.params['treated:post']:.2f} "
      f"(95% CI {fit.conf_int().loc['treated:post', 0]:.2f} ~ {fit.conf_int().loc['treated:post', 1]:.2f})")
print("解讀：交互項接近真值 -8；DiD 以 control 組的前後變化代表『若無政策的共同趨勢』，相減後即得政策淨效果。")

## 題目 5 解答

In [ ]:
# 口罩政策 DiD：部分縣市實施口罩令（treated），outcome 為每週病例成長率
rng = np.random.default_rng(1205)
_rows = []
TRUE_EFFECT = -0.18   # 口罩令讓成長率下降 0.18
for reg in range(160):
    treated = 1 if reg < 80 else 0
    base = rng.normal(0.30, 0.05)
    for post in (0, 1):
        g = base - 0.05 * post + TRUE_EFFECT * (treated * post) + rng.normal(0, 0.04)
        _rows.append({"region": reg, "treated": treated, "post": post, "growth": g})
mask = pd.DataFrame(_rows)
print(f"DiD 資料：{mask['region'].nunique()} 縣市 × 2 期，真值效果 = {TRUE_EFFECT}")

fit = smf.ols("growth ~ treated * post", mask).fit()
did = fit.params["treated:post"]
print(f"DiD（口罩令效果）= {did:.3f} "
      f"(95% CI {fit.conf_int().loc['treated:post', 0]:.3f} ~ {fit.conf_int().loc['treated:post', 1]:.3f})")
print("解讀：交互項為負且接近真值 -0.18 → 口罩令使每週病例成長率下降約 0.18。")

## 題目 6 解答

In [ ]:
# 吸菸與疾病：年齡是干擾因子（年長者較常吸菸、也較易生病）
rng = np.random.default_rng(1206)
n = 3000
age = rng.integers(20, 80, n)
smoke = rng.binomial(1, 1 / (1 + np.exp(-(-2.5 + 0.05 * age))))
logit = -4.5 + 0.05 * age + 0.8 * smoke     # smoke 真實 log-OR = 0.8
disease = rng.binomial(1, 1 / (1 + np.exp(-logit)))
dat = pd.DataFrame({"age": age, "smoke": smoke, "disease": disease})
print(f"n={n}，吸菸率={smoke.mean():.1%}，疾病率={disease.mean():.1%}（smoke 真實 log-OR=0.8）")

crude = smf.logit("disease ~ smoke", data=dat).fit(disp=0)
adj = smf.logit("disease ~ smoke + age", data=dat).fit(disp=0)
print(f"crude    smoke 係數(log-OR) = {crude.params['smoke']:.3f}  → OR={np.exp(crude.params['smoke']):.2f}")
print(f"adjusted smoke 係數(log-OR) = {adj.params['smoke']:.3f}  → OR={np.exp(adj.params['smoke']):.2f}  (真值 0.8)")
print("解讀：crude 高估（年齡同時推高吸菸與疾病，正向干擾）；校正年齡後 smoke 係數回到真值附近。")

## 題目 7 解答

In [ ]:
# COVID-19 治療的傾向分數配對：病情越重越可能被治療（干擾），但治療其實有益
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
rng = np.random.default_rng(1207)
n = 2000
severity = rng.uniform(0, 1, n)
treated = rng.binomial(1, 0.15 + 0.7 * severity)  # 越重越常被治療（confounding by indication；保留重疊）
TRUE_EFFECT = -0.15                              # 治療真正讓死亡率下降 0.15
death_p = (0.10 + 0.75 * severity + TRUE_EFFECT * treated).clip(0.01, 0.99)
death = rng.binomial(1, death_p)
cov = pd.DataFrame({"severity": severity, "treated": treated, "death": death})
print(f"n={n}，治療比例={treated.mean():.1%}，真值治療效果(死亡率差)={TRUE_EFFECT}")

naive = cov.loc[cov.treated == 1, "death"].mean() - cov.loc[cov.treated == 0, "death"].mean()
ps = LogisticRegression(max_iter=1000).fit(cov[["severity"]], cov["treated"]).predict_proba(cov[["severity"]])[:, 1]
cov["ps"] = ps
tr = cov[cov.treated == 1]; ct = cov[cov.treated == 0]
nn = NearestNeighbors(n_neighbors=1).fit(ct[["ps"]].values)
_, idx = nn.kneighbors(tr[["ps"]].values)
matched_ctrl_death = ct["death"].values[idx.ravel()]
att = tr["death"].mean() - matched_ctrl_death.mean()
print(f"naive 死亡率差 = {naive:+.3f}（偏誤：重症者較常被治療，治療看似有害）")
print(f"傾向分數配對後 ATT = {att:+.3f}（真值 {-0.15}）")
print("解讀：配對讓治療組與對照組的病情(severity)分布相近，去除 confounding by indication 後，治療的保護效果才顯現。")

## 題目 8 解答

In [ ]:
# 工具變數 IV：暴露受未觀測干擾 U 影響（內生），Z 為工具變數（挑戰題）
rng = np.random.default_rng(1208)
n = 3000
U = rng.normal(0, 1, n)                 # 未觀測干擾
Z = rng.normal(0, 1, n)                 # 工具變數：影響暴露、不直接影響結果
exposure = 0.6 * Z + 0.7 * U + rng.normal(0, 1, n)
TRUE_EFFECT = 1.5
outcome = TRUE_EFFECT * exposure + 1.2 * U + rng.normal(0, 1, n)
iv = pd.DataFrame({"Z": Z, "exposure": exposure, "outcome": outcome})
print(f"n={n}，暴露真實因果效果 = {TRUE_EFFECT}（naive OLS 會因 U 而高估）")

naive = smf.ols("outcome ~ exposure", iv).fit()
stage1 = smf.ols("exposure ~ Z", iv).fit()
iv["exp_hat"] = stage1.fittedvalues
stage2 = smf.ols("outcome ~ exp_hat", iv).fit()
print(f"naive OLS 係數 = {naive.params['exposure']:.3f}（被未觀測干擾 U 高估）")
print(f"IV (2SLS) 係數 = {stage2.params['exp_hat']:.3f}（真值 1.5）")
print(f"第一階段 Z→exposure 係數 = {stage1.params['Z']:.3f}, F≈{stage1.fvalue:.0f}（工具夠強）")
print("解讀：IV 利用只透過暴露影響結果的外生變異(Z)來估計因果效果；")
print("Z 需滿足：(1) 與暴露相關(相關性)、(2) 只透過暴露影響結果(排除限制)、(3) 與 U 無關。")

## 題目 9：食物中毒的 AR/PAR 與清消措施的 DiD

某國小營養午餐爆發疑似食物中毒事件（本題資料為教學用合成情境，並非真實案例）。衛生單位懷疑當日供應的「涼拌小黃瓜」為可疑污染來源，稽核後要求團膳廠商加強清消措施。

**第一部分：AR / PAR（歸因風險）**

1. 用下方 2×2 表的 `a, b, c, d` 計算「吃了涼拌小黃瓜」與「沒吃」兩組的侵襲率
2. 計算風險比（RR）與歸因風險（AR，risk difference）
3. 用 Levin 公式計算族群歸因風險百分比（PAF）：`Pe * (RR - 1) / (1 + Pe * (RR - 1))`，其中 Pe 為全體中「有吃該菜色」的比例
4. 解讀 PAF：如果把這道菜從菜單中移除，理論上能減少多少比例的病例？

**第二部分：DiD（清消措施效果）**

5. 用下方面板資料，以 `smf.ols("cases ~ treated + post + treated:post", data=food).fit(cov_type="HC3")` 估計清消措施的介入效果（HC3 穩健標準誤）
6. `treated:post` 交互項代表什麼？和資料中設定的真值效果相比如何？
7. DiD 方法能成立的關鍵前提是什麼（平行趨勢假設）？如果介入前 treated 組和 control 組的病例趨勢本來就不平行，結果會如何被扭曲？

In [ ]:
# 食物中毒 2x2：吃了涼拌小黃瓜 vs 沒吃 × 發病 vs 未發病（教學用合成數據，非真實案例）
a, b = 90, 30    # 吃了嫌疑菜色：發病 / 未發病
c, d = 20, 180   # 沒吃嫌疑菜色：發病 / 未發病
print(f"吃了嫌疑菜色：{a + b} 人（發病 {a}，未發病 {b}）")
print(f"沒吃嫌疑菜色：{c + d} 人（發病 {c}，未發病 {d}）")

# 清消措施 DiD 面板：部分學校加強清消（treated），比較措施前後每日通報病例數（教學用合成數據）
rng = np.random.default_rng(1209)
_rows = []
TRUE_EFFECT = -4.0   # 加強清消措施讓每日通報病例數平均減少 4 例
for school in range(120):
    treated = 1 if school < 60 else 0
    base = rng.normal(10, 2)
    for post in (0, 1):
        cases = base - 1 * post + TRUE_EFFECT * (treated * post) + rng.normal(0, 1.5)
        _rows.append({"school": school, "treated": treated, "post": post, "cases": cases})
food = pd.DataFrame(_rows)
print(f"DiD 資料：{food['school'].nunique()} 校 × 2 期，真值效果 = {TRUE_EFFECT} 例/日")

In [ ]:
# 食物中毒 AR/PAR：吃了涼拌小黃瓜 vs 沒吃
risk_exp = a / (a + b)
risk_unexp = c / (c + d)
RR = risk_exp / risk_unexp
AR = risk_exp - risk_unexp
Pe = (a + b) / (a + b + c + d)
PAF = Pe * (RR - 1) / (1 + Pe * (RR - 1))

print("=== 涼拌小黃瓜暴露：AR / PAR ===")
print(f"吃了嫌疑菜色侵襲率：{risk_exp:.1%}")
print(f"沒吃嫌疑菜色侵襲率：{risk_unexp:.1%}")
print(f"風險比 RR = {RR:.2f}")
print(f"歸因風險 AR = {AR:.3f}")
print(f"Pe（全體中有吃該菜色比例）= {Pe:.1%}")
print(f"族群歸因風險百分比 PAF（Levin 公式）= {PAF:.1%}")
print(f"\u2192 若能完全移除這道菜，理論上可減少約 {PAF:.0%} 的病例")

# 清消措施 DiD
fit = smf.ols("cases ~ treated + post + treated:post", data=food).fit(cov_type="HC3")
did_coef = fit.params["treated:post"]
did_p = fit.pvalues["treated:post"]
ci_lo, ci_hi = fit.conf_int().loc["treated:post"]
print("\n=== 清消措施 DiD ===")
print(f"treated:post 係數 = {did_coef:.2f}（95% CI {ci_lo:.2f} ~ {ci_hi:.2f}, p = {did_p:.4f}）")
print(f"真值效果 = {TRUE_EFFECT}")
print("\u2192 交互項為負且接近真值，顯示加強清消措施後每日通報病例數顯著下降")
print("\u2192 平行趨勢假設：DiD 成立的前提是若無介入，treated 與 control 組的病例趨勢應維持平行；")
print("   若介入前兩組本來就有不同趨勢，DiD 會把趨勢差誤判為介入效果")